In [1]:
import numpy as np
import matplotlib.pyplot as plt
import nifty8 as ift
from phase_II.utils.helpers import Stress, visualize_stress, usual_plot
from phase_I.utils.config_jupyter_notebooks import *
%matplotlib tk

nrt_strain_values = np.loadtxt("../../data/data_txt/num_rel_template_strain_values.txt") * 1e19
nrt_time_values = np.loadtxt("../../data/data_txt/num_rel_template_time_values.txt") - zero_time

Important variables: 
		signal_strip_time, signal_strip_strain 
		signal_strip_strain_tapered
		strain
		time_domain_strip
		N


In [17]:
from phase_II.nifty_re_playground.helpers import get_sample_data

S_mat_inference, t_dual_inference, f_inference = unpickle_me_this("../wigner_result_pipe_2.pickle")
time_tmp, _ = get_sample_data()
t_dual_inference = t_dual_inference + min(time_tmp)  # In my head the grav wave starts at 16.4 not at 1.4

In [18]:
visualize_stress(S_mat_inference, rows=f_inference, cols=t_dual_inference)

		Rows must be in ascending order for visualization purposes but they are not, assuming a priori standard DFT order and moving DC to the middle


## Invert without smoothing

In [4]:
def invert_wigner_function(S_mat, frequency_array, time_domain, xi_tilde_0=None):
    f = frequency_array
    t_vol = time_domain.scalar_dvol

    r_dom = time_domain
    h_dom = r_dom.get_default_codomain()
    FFT_forward = ift.FFTOperator(domain=(h_dom, r_dom), space=1) * (1/t_vol)

    S_mat_field = ift.Field(domain=ift.DomainTuple.make((h_dom, r_dom)), val=S_mat)

    Sigma_f_q = FFT_forward(S_mat_field).val

    # k: 1D frequency array length K
    # Sigma: 2D array shape (K, K) with axes (k1, k2)
    k_half = 0.5 * f               # desired first-axis locations
    # find nearest index in k for each k_half
    idx_rows = np.argmin(np.abs(f[:, None] - k_half[None, :]), axis=0)  # shape (K,)
    # pick one element per column j: Sigma[idx_rows[j], j]
    vec = Sigma_f_q[idx_rows, np.arange(len(f))]    # shape (K,)

    if xi_tilde_0 is not None:
        vec = vec / xi_tilde_0.conj()

    tmp = np.fft.ifft(vec, norm="ortho")

    return tmp

In [5]:
N = len(t_dual_inference)
dt = t_dual_inference[1]-t_dual_inference[0]
time_dom = ift.RGSpace(shape=(N), distances=dt)

In [6]:
reconstructed_xi = invert_wigner_function(S_mat=S_mat_inference, frequency_array=f_inference, time_domain=time_dom)

In [7]:
plt.plot(t_dual_inference, reconstructed_xi)
usual_plot(title="Reconstructed xi field without smoothing of Wigner function")

/Users/iason/PycharmProjects/stability-of-submoons/.venv/lib/python3.12/site-packages/matplotlib/cbook/__init__.py:1345: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(x, float)


## Implement some sort of smoothing

In [8]:
visualize_stress(S_mat_inference, rows=f_inference, cols=t_dual_inference)

		Rows must be in ascending order for visualization purposes but they are not, assuming a priori standard DFT order and moving DC to the middle


In [15]:
from scipy.ndimage import gaussian_filter
sm = gaussian_filter(S_mat_inference, sigma=10.0)   # sigma ~ 0.5..3 blur radius in pixels

# threshhold for background subtraction

# mean_sm = np.mean(sm).real
# print("Smoothed mean value: ", mean_sm)
# thresh = mean_sm*10
# sm[np.where(sm.real<thresh)] = 0

print("Applying threshhold: ", thresh)

Applying threshhold:  524.1131642967863


In [16]:
visualize_stress(sm, rows=f_inference, cols=t_dual_inference)

		Rows must be in ascending order for visualization purposes but they are not, assuming a priori standard DFT order and moving DC to the middle


In [11]:
reconstructed_xi = invert_wigner_function(S_mat=sm, frequency_array=f_inference, time_domain=time_dom)

In [12]:
plt.plot(t_dual_inference, reconstructed_xi/max(reconstructed_xi)*max(nrt_strain_values), label="Reconstructed xi field (scaled)")
plt.plot(nrt_time_values, nrt_strain_values)
usual_plot(title="Reconstructed xi field after Gaussian smoothing of Wigner function")